# 01 更新数据（美股 + 新加坡，一次完成）

**用法**：重启内核 → Run All。需要先安装 `yfinance`（`pip install yfinance`）。

一次下载 9 只美股 ETF + `ES3.SI`、`A35.SI` + `SGD=X`（汇率），起始日 2011-10-01，`auto_adjust=False`，然后**直接覆盖**两个数据文件：

| 文件 | 内容 | 谁在用 |
|---|---|---|
| `data/raw/etf_daily.csv` | 全部标的，yfinance 原始多层表头格式 | 02、05、06、07、08 |
| `data/raw/sti_daily.csv` | 一行一个 ES3 交易日：ES3、A35（`bond_` 开头）、汇率 `usdsgd` | 03、15、12 |

- **覆盖前**把旧文件备份为 `etf_daily_backup.csv`、`sti_daily_backup.csv`（每次覆盖上一份备份）。
- 尚未收盘时下载到的"今天"是盘中价格，会被丢弃：美股按纽约时间 16:30，新加坡按新加坡时间 17:30。
- 9 只美股 ETF 有任何一只下载失败，两个文件都不会被覆盖；新加坡标的下载失败时只跳过 `sti_daily.csv`。


In [ ]:
from pathlib import Path
import os
import json

def find_pipeline_dir():
    """支持从项目根目录、pipeline 目录及项目内子目录启动。"""
    roots = ([Path(os.environ['ETF_PROJECT_ROOT']).expanduser()] if os.environ.get('ETF_PROJECT_ROOT') else [])
    roots += [Path.cwd(), *Path.cwd().parents]
    for root in roots:
        for candidate in (root / 'pipeline', root):
            if (candidate / '00_engine.ipynb').is_file():
                return candidate.resolve()
    raise FileNotFoundError('找不到 pipeline 文件夹。请将 PyCharm 工作目录设为项目根目录，或设置 ETF_PROJECT_ROOT。')

PIPELINE_DIR = find_pipeline_dir()

def load_notebook_definitions(filename):
    """独立命名空间加载函数；不执行被加载 notebook 的入口单元格。"""
    path = next(iter(sorted(PIPELINE_DIR.rglob(filename))), PIPELINE_DIR / filename)   # 在 pipeline 各子文件夹中查找
    notebook = json.loads(path.read_text(encoding='utf-8'))
    namespace = {'__name__': 'pipeline_module'}
    for number, item in enumerate(notebook['cells']):
        if item['cell_type'] == 'code' and 'pipeline-entry' not in item.get('metadata', {}).get('tags', []):
            exec(compile(''.join(item['source']), f'{path}:cell{number}', 'exec'), namespace)
    return namespace

_engine = load_notebook_definitions('00_engine.ipynb')
for _name, _value in _engine.items():
    if not _name.startswith('_'):
        globals()[_name] = _value


In [ ]:
import shutil

DATA_DIR = PIPELINE_DIR.parent / 'data' / 'raw'
ETF_CSV, STI_CSV = DATA_DIR / 'etf_daily.csv', DATA_DIR / 'sti_daily.csv'
SG = {'ES3': 'ES3.SI', 'A35': 'A35.SI', 'FX': 'SGD=X'}


def drop_unfinished(df, tz, close_minutes, label):
    """当地时间收盘之前，"今天"这一行是盘中价格，丢弃。"""
    now = pd.Timestamp.now(tz=tz)
    if len(df) and df.index[-1].date() >= now.date() and now.hour * 60 + now.minute < close_minutes:
        print(f'{label}：丢弃未收盘的当日数据 {df.index[-1].date()}（当地时间 {now:%H:%M} 尚未收盘）')
        df = df.iloc[:-1]
    return df


def has(data, t):
    return ('Adj Close', t) in data and not data['Adj Close'][t].dropna().empty


def download_all(start='2011-10-01'):
    import yfinance as yf
    data = yf.download(TICKS + list(SG.values()), start=start, auto_adjust=False)
    if data.empty:
        raise ValueError('下载结果为空，未覆盖任何文件。')
    missing = [t for t in TICKS if not has(data, t)]
    if missing:
        raise ValueError(f'下载缺少美股标的：{missing}；未覆盖任何文件。')
    return data


def build_etf_daily(data):
    return drop_unfinished(data, 'America/New_York', 16 * 60 + 30, 'etf_daily（美股）')


def build_sti_daily(data):
    """与 03 号 load_sti() 读取的格式一致：一行一个 ES3 交易日。"""
    def ohlc(t, prefix=''):
        return pd.DataFrame({f'{prefix}open': data['Open'][t], f'{prefix}high': data['High'][t],
                             f'{prefix}low': data['Low'][t], f'{prefix}close': data['Close'][t],
                             f'{prefix}adj_close': data['Adj Close'][t], f'{prefix}volume': data['Volume'][t]})
    out = ohlc(SG['ES3']).dropna(subset=['close'])
    out = out.join(ohlc(SG['A35'], 'bond_').reindex(out.index))     # A35 缺失的日子留空，03 号读取时按"价格不变"处理
    fx = data['Close'][SG['FX']].dropna()
    out['usdsgd'] = fx.reindex(out.index.union(fx.index)).ffill().reindex(out.index)   # 取不晚于当天的最近汇率
    out.index.name = 'Date'
    return drop_unfinished(out, 'Asia/Singapore', 17 * 60 + 30, 'sti_daily（新加坡）')


def overwrite(df, target):
    if target.is_file():
        shutil.copy2(target, target.with_name(target.stem + '_backup.csv'))
    df.to_csv(target)


def main(download=True):
    if not download:
        print('未下载（DOWNLOAD = False）。')
        return None
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    data = download_all()

    etf = build_etf_daily(data)
    overwrite(etf, ETF_CSV)
    last_us = etf['Adj Close'][TICKS].dropna().index.max().date()
    print(f'已覆盖 {ETF_CSV.name}（旧文件备份为 {ETF_CSV.stem}_backup.csv）| 美股最后完整交易日: {last_us}')

    sti = None
    lost = [t for t in SG.values() if not has(data, t)]
    if lost:
        print(f'⚠ 新加坡标的 {lost} 没有下载到，{STI_CSV.name} 保持不变。')
    else:
        sti = build_sti_daily(data)
        overwrite(sti, STI_CSV)
        print(f'已覆盖 {STI_CSV.name}（旧文件备份为 {STI_CSV.stem}_backup.csv）| '
              f'{sti.index[0].date()} ~ {sti.index[-1].date()} | A35 有数据的天数: {int(sti["bond_close"].notna().sum())}')
    return {'etf_daily': etf, 'sti_daily': sti}


In [ ]:
DOWNLOAD = True      # 改为 False 则不下载
results = main(download=DOWNLOAD)
